#Import

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

#Create Flag Parameter

In [0]:
# Check whether the Gold dimension table already exists
if spark.catalog.tableExists("store_catalog.gold.dim_erp_LOC_A101"):
    incremental_flag = "1"
else:
    incremental_flag = "0"

In [0]:
print(incremental_flag)

1


#Create Dimension erp_LOC_A101

###Fetch Relative Columns

In [0]:
# Load distinct customer records from the Silver layer
df_scr = spark.sql('''
        SELECT DISTINCT * FROM parquet.`abfss://silver@storenapatdatalake.dfs.core.windows.net/erp_LOC_A101`
    ''')


In [0]:
df_scr.limit(10).display()

customer_number,country,record_priority
AW00011079,Australia,1
AW00012066,United States,1
AW00012094,United States,1
AW00012103,United States,1
AW00012193,Canada,1
AW00012334,United Kingdom,1
AW00012577,Australia,1
AW00012586,Australia,1
AW00012972,Canada,1
AW00013231,Canada,1


###dim_erp_LOC_A101 - Initial and Incremental

In [0]:
# Read existing Gold records or create an empty DataFrame for the initial load
if spark.catalog.tableExists("store_catalog.gold.dim_erp_LOC_A101") == True:
    df_sink = spark.sql('''
            SELECT
                erp_LOC_A101_surrogate_key,
                customer_number,
                country
            FROM
                store_catalog.gold.dim_erp_LOC_A101
        ''')

else:
    df_sink = spark.sql('''
            SELECT
                1 as erp_LOC_A101_surrogate_key,
                customer_number,
                country
            FROM
                parquet.`abfss://silver@storenapatdatalake.dfs.core.windows.net/erp_LOC_A101`
            WHERE
                1 = 0
        ''')


###Filtering new records and old records

In [0]:
# Match Silver records with existing Gold records by customer number
df_filter = df_scr.join(df_sink, df_scr["customer_number"] == df_sink["customer_number"], "left")\
        .select(df_scr["customer_number"],df_scr["country"],df_sink["erp_LOC_A101_surrogate_key"]
    )

**df_filter_old**

In [0]:
# Separate existing customer records
df_filter_old = df_filter.filter(col("erp_LOC_A101_surrogate_key").isNotNull())

**df_filter_new**

In [0]:
# Select new customers that require surrogate keys
df_filter_new = df_filter.filter(col("erp_LOC_A101_surrogate_key").isNull())\
    .select(df_scr["customer_number"],df_scr["country"]
    )

In [0]:
df_filter_new.limit(10).display()

customer_number,country


###Create Surrogate Key

**Fetch the max Surrogate Key from existing table**

In [0]:
# Retrieve the next available surrogate key
if (incremental_flag == "0"):
    max_value = 1
else:
    max_value_df = spark.sql('''SELECT MAX(erp_LOC_A101_surrogate_key) FROM store_catalog.gold.dim_erp_LOC_A101''')
    max_value = max_value_df.collect()[0][0]+1

**Create Surrogate Key column and Add the max surrogate key**

In [0]:
# Assign unique surrogate keys to new customer records
df_filter_new = df_filter_new.withColumn("erp_LOC_A101_surrogate_key",max_value + monotonically_increasing_id())
df_filter_new.limit(10).display()

customer_number,country,erp_LOC_A101_surrogate_key


###Create Final DF - df_filter_old + df_filter_new

In [0]:
# Combine new customers with existing customer records
df_final = df_filter_new.union(df_filter_old)

In [0]:
df_final.limit(10).display()

customer_number,country,erp_LOC_A101_surrogate_key
AW00011079,Australia,1
AW00012066,United States,2
AW00012094,United States,3
AW00012103,United States,4
AW00012193,Canada,5
AW00012334,United Kingdom,6
AW00012577,Australia,7
AW00012586,Australia,8
AW00012972,Canada,9
AW00013231,Canada,10


# SCD TYPE - 1 (UPSERT)

In [0]:
from delta.tables import DeltaTable

In [0]:
if spark.catalog.tableExists("store_catalog.gold.dim_erp_LOC_A101"):
    # Incremental Run
    # Merge customer records into the Gold dimension table
    delta_tbl = DeltaTable.forPath(spark,"abfss://gold@storenapatdatalake.dfs.core.windows.net/dim_erp_LOC_A101")

    delta_tbl.alias("trg").merge(df_final.alias("src"),"trg.erp_LOC_A101_surrogate_key = src.erp_LOC_A101_surrogate_key")\
                            .whenMatchedUpdateAll()\
                            .whenNotMatchedInsertAll()\
                            .execute()

else: 
    # Initial Run
    # Create the Gold dimension table during the initial load
    df_final.write.format("delta")\
        .mode("overwrite")\
        .option("path","abfss://gold@storenapatdatalake.dfs.core.windows.net/dim_erp_LOC_A101")\
        .saveAsTable("store_catalog.gold.dim_erp_LOC_A101")


In [0]:
%sql
SELECT * FROM store_catalog.gold.dim_erp_LOC_A101  ORDER BY erp_LOC_A101_surrogate_key DESC LIMIT 10

customer_number,country,erp_LOC_A101_surrogate_key
AW00029375,Germany,18484
AW00029261,United States,18483
AW00028830,United States,18482
AW00027911,United States,18481
AW00027908,Germany,18480
AW00027772,France,18479
AW00027455,United States,18478
AW00027120,Germany,18477
AW00026662,Australia,18476
AW00026524,United States,18475


In [0]:
%sql
SELECT COUNT(*) FROM store_catalog.gold.dim_erp_LOC_A101 

count(1)
18484
